In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("init_load_flag", "0")

In [0]:
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

#### Reading data

In [0]:
df = spark.read.table("databricks_pipeline_cata.silver.customers")

In [0]:
df.display()

#### Removing duplicates based on primary key

In [0]:
df = df.dropDuplicates(subset=["customer_id"])

# Dividing old vs new records

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''SELECT DimCustomerKey, customer_id, create_date, update_date
                           FROM databricks_pipeline_cata.gold.DimCustomers
                       ''')
else:
    df_old = spark.sql('''SELECT 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date
                            FROM databricks_pipeline_cata.silver.customers WHERE 1=0''')

In [0]:
df_old.display()

#### Renaming columns of df_old

In [0]:
df_old = df_old.withColumnRenamed("DimCustomerKey", "old_DimCustomerKey")\
                .withColumnRenamed("create_date", "old_create_date")\
                .withColumnRenamed("update_date", "old_update_date")\
                .withColumnRenamed("customer_id", "old_customer_id")

#### Applying join with old records

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, "left")

In [0]:
df_join.display()

#### Separating new vs old records

In [0]:
df_new = df_join.filter(col("old_DimCustomerKey").isNull())

In [0]:
df_old = df_join.filter(col("old_DimCustomerKey").isNotNull())

#### Preparing df_old

In [0]:
# Dropping all the columns which are not required
df_old = df_old.drop("old_customer_id", "old_update_date")

# Renaming "old_create_date" column to "create_date" and "old_DimCustomerKey" to "DimCustomerKey"
df_old = df_old.withColumnRenamed("old_DimCustomerKey", "DimCustomerKey")

df_old = df_old.withColumnRenamed("old_create_date", "create_date")
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))

# Recreating "update_date" with the current timestamp
df_old = df_old.withColumn("update_date", current_timestamp())

In [0]:
df_old.display()

#### Preparing df_new

In [0]:
# Dropping columns that are not required
df_new = df_new.drop("old_DimCustomerKey", "old_customer_id", "old_create_date", "old_update_date")

# Recreating "current_date", "update_date" column with the current_timestamp
df_new = df_new.withColumn("create_date", current_timestamp())\
                .withColumn("update_date", current_timestamp())

In [0]:
df_new.display()

#### Surrogate keys - From 1
- We create dimension key and use that key for foreign key relation

In [0]:
df_new = df_new.withColumn("dimCustomerKey", monotonically_increasing_id()+lit(1))

In [0]:
df_new.display()

#### Adding max surrogate key of df_old

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_surr = spark.sql("SELECT max(dimCustomerKey) as max_surrogate_key FROM databricks_pipeline_cata.gold.DimCustomers")
    max_surrogate_key = df_surr.collect()[0]["max_surrogate_key"] # Converting df_surr to max_surrogate_key

In [0]:
df_new = df_new.withColumn("dimCustomerKey", col("dimCustomerKey") + max_surrogate_key)

#### Union of df_old and df_new

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()

#### SCD Type -1

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks_pipeline_cata.gold.DimCustomers"): # can also use 'init_load_flag==0
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@databrickspipeline07.dfs.core.windows.net/DimCustomers")

    dlt_obj.alias("trg").merge(df_final.alias("src"), "trg.dimCustomerKey = src.dimCustomerKey")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()

else:
    df_final.write.mode("overwrite")\
        .format("delta")\
        .option("path", "abfss://gold@databrickspipeline07.dfs.core.windows.net/DimCustomers")\
        .saveAsTable("databricks_pipeline_cata.gold.DimCustomers")  

In [0]:
%sql
SELECT * FROM databricks_pipeline_cata.gold.dimcustomers

In [0]:
init_load_flag = 1
dbutils.widgets.remove("init_load_flag")